# Used Car Price Prediction - Google Colab Integration

This Colab notebook shows how to train the model, evaluate results, and save a trained model using the same features as the local project. It also explains how to use the Streamlit UI locally after training.

## 1. Install dependencies

Run the following cell to install the required Python packages in Colab.

In [1]:
!pip install -q pandas scikit-learn xgboost joblib

'c:\Users\krish\OneDrive\Desktop\~Code\used_car_price_prediction\.venv-1\Scripts\pip.exe' was blocked by your organization's Device Guard policy.
Contact your support person for more info.


## 2. Upload the dataset

Use the file upload widget below to upload `used_car_price_prediction.csv` from your local machine.

In [ ]:
from google.colab import files
uploaded = files.upload()
import pandas as pd
csv_file = list(uploaded.keys())[0] if uploaded else 'used_car_price_prediction.csv'
df = pd.read_csv(csv_file)
print('Dataset loaded:', df.shape)
print(df.head())

## 3. Train advanced models and ensemble

This cell trains Random Forest, XGBoost, and a voting ensemble. It also computes training and validation performance to check for overfitting.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, VotingRegressor, StackingRegressor
from sklearn.linear_model import RidgeCV
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

def add_brand_feature(df):
    df = df.copy()
    bins = [-1, 3, 5, 7, 10, 15, 20, 30, 45, 60, 1000]
    labels = [
        'Maruti', 'Hyundai', 'Honda', 'Toyota', 'Ford', 'Mahindra', 'Tata', 'BMW', 'Mercedes', 'Audi'
    ]
    df['Brand'] = pd.cut(df['Present_Price'], bins=bins, labels=labels)
    df['Brand'] = df['Brand'].fillna(labels[0]).astype(str)
    return df

df = add_brand_feature(df)
X = df.drop('Price', axis=1)
y = df['Price']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

models = {
    'RandomForest': Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42))
    ]),
    'XGBoost': Pipeline([
        ('preprocessor', preprocessor),
        ('model', XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42, objective='reg:squarederror', verbosity=0))
    ]),
    'VotingEnsemble': Pipeline([
        ('preprocessor', preprocessor),
        ('model', VotingRegressor([
            ('rf', RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)),
            ('xgb', XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42, objective='reg:squarederror', verbosity=0)),
        ]))
    ]),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)
    results[name] = {
        'train_r2': r2_score(y_train, train_pred),
        'val_r2': r2_score(y_val, val_pred),
        'train_mae': mean_absolute_error(y_train, train_pred),
        'val_mae': mean_absolute_error(y_val, val_pred),
        'train_rmse': mean_squared_error(y_train, train_pred, squared=False),
        'val_rmse': mean_squared_error(y_val, val_pred, squared=False),
        'overfit_gap': r2_score(y_train, train_pred) - r2_score(y_val, val_pred),
    }

best_name = max(results, key=lambda x: results[x]['val_r2'])
print('Best model:', best_name)
for name, metrics in results.items():
    print(name, metrics)

best_model = models[best_name]
joblib.dump(best_model, 'ensemble_model_colab.pkl')
print('Saved model to ensemble_model_colab.pkl')

## 4. Download the trained model

After training, download `ensemble_model_colab.pkl` to use it locally or for further evaluation.

In [ ]:
from google.colab import files
files.download('ensemble_model_colab.pkl')

## 5. Use the local Streamlit UI

The Streamlit UI in this project runs locally from `app/ui.py`. After training in Colab, copy the saved model file into the local `models/` folder and run the app with:

```bash
streamlit run app/ui.py
```

For Colab-only exploration, training and evaluation are done in the notebook, while the UI remains a local Streamlit web app.